# 04 — Ecommify: ETL PostgreSQL → MongoDB (Sincronización)

**Prerrequisito**: Notebooks 01, 02, 03 ejecutados.

Este notebook ejecuta el ETL incremental PG → MongoDB y verifica consistencia entre sistemas.

## 1. Setup y conexiones

In [ ]:
!pip install --quiet psycopg2-binary 'pymongo[srv]' pandas
import psycopg2, getpass
from psycopg2.extras import RealDictCursor
from pymongo import MongoClient, UpdateOne
from datetime import datetime, timezone
import pandas as pd
from google.colab import userdata

SUPABASE_URI = userdata.get('SUPABASE_URI')
ATLAS_URI    = userdata.get('ATLAS_URI')

pg = psycopg2.connect(SUPABASE_URI, cursor_factory=RealDictCursor,
                      options='-c statement_timeout=600000')
mc = MongoClient(ATLAS_URI)
db = mc['ecommify']
print('Conexiones OK')

## 2. ETL incremental orders_summary

In [ ]:
# Determinar último timestamp sincronizado
last_doc = db.orders_summary.find_one(sort=[('etl_updated_at', -1)])
last_ts = last_doc['etl_updated_at'] if last_doc else datetime(1970,1,1, tzinfo=timezone.utc)
print(f'Último ETL: {last_ts}')

SQL = """
SELECT o.order_id::TEXT, o.order_status, o.order_purchase_timestamp,
       c.customer_id::TEXT, c.customer_state, c.customer_city,
       COALESCE(
           json_agg(json_build_object(
               'product_id', oi.product_id::TEXT, 'seller_id', oi.seller_id::TEXT,
               'price', oi.price::FLOAT, 'freight_value', oi.freight_value::FLOAT
           ) ORDER BY oi.order_item_id) FILTER (WHERE oi.order_id IS NOT NULL),
           '[]'::json
       ) AS items,
       COALESCE(SUM(op.payment_value), 0)::FLOAT AS payment_total,
       (ARRAY_AGG(op.payment_type ORDER BY op.payment_value DESC) FILTER (WHERE op.payment_type IS NOT NULL))[1] AS payment_type_main
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
LEFT JOIN order_items oi ON oi.order_id = o.order_id AND oi.order_purchase_timestamp = o.order_purchase_timestamp
LEFT JOIN order_payments op ON op.order_id = o.order_id AND op.order_purchase_timestamp = o.order_purchase_timestamp
WHERE o.updated_at > %s
GROUP BY o.order_id, o.order_status, o.order_purchase_timestamp, c.customer_id, c.customer_state, c.customer_city
"""

with pg.cursor() as cur:
    cur.execute(SQL, (last_ts,))
    rows = cur.fetchall()
print(f'Filas a sincronizar: {len(rows):,}')

if rows:
    now = datetime.now(timezone.utc)
    ops = []
    for r in rows:
        doc = {
            '_id': r['order_id'], 'status': r['order_status'],
            'purchase_date': r['order_purchase_timestamp'],
            'customer': {'customer_id': r['customer_id'],
                         'state': r['customer_state'], 'city': r['customer_city']},
            'items': r['items'] if r['items'] else [],
            'payment_total': r['payment_total'] or 0.0,
            'payment_type_main': r['payment_type_main'],
            'review_score': None, 'etl_updated_at': now,
        }
        ops.append(UpdateOne({'_id': doc['_id']}, {'$set': doc}, upsert=True))
        if len(ops) >= 500:
            result = db.orders_summary.bulk_write(ops, ordered=False)
            print(f'  Batch: {result.upserted_count} insertados, {result.modified_count} actualizados')
            ops = []
    if ops:
        result = db.orders_summary.bulk_write(ops, ordered=False)
        print(f'  Batch final: {result.upserted_count} insertados, {result.modified_count} actualizados')
else:
    print('Sin cambios desde el último ETL.')

## 3. Verificar consistencia entre sistemas

In [ ]:
print('=== Verificación de consistencia PG ↔ MongoDB ===')

with pg.cursor() as cur:
    cur.execute("SELECT order_status, COUNT(*) FROM orders GROUP BY order_status ORDER BY 2 DESC")
    pg_stats = {r['order_status']: r['count'] for r in cur.fetchall()}

mdb_stats = {}
for s in db.orders_summary.aggregate([{'$group': {'_id': '$status', 'count': {'$sum': 1}}}]):
    mdb_stats[s['_id']] = s['count']

print(f'  {'Status':20s} {'PostgreSQL':>12s} {'MongoDB':>12s} {'Diferencia':>12s}')
print('  ' + '-' * 60)
all_statuses = sorted(set(pg_stats) | set(mdb_stats))
for s in all_statuses:
    pg_n = pg_stats.get(s, 0)
    mdb_n = mdb_stats.get(s, 0)
    diff = abs(pg_n - mdb_n)
    flag = '✓' if diff == 0 else f'⚠ diff={diff}'
    print(f'  {s:20s} {pg_n:>12,} {mdb_n:>12,} {flag:>12s}')

# Total
with pg.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM orders')
    pg_total = cur.fetchone()[0]
mdb_total = db.orders_summary.count_documents({})
print(f'\n  Total PostgreSQL: {pg_total:,}')
print(f'  Total MongoDB:    {mdb_total:,}')
print(f'  Cobertura ETL:    {mdb_total/pg_total*100:.1f}%')

pg.close(); mc.close()
print('\nNotebook 04 completado.')